# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ayazahmedbscsf23-art/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [3]:
import os
DATA_PATH = "content_refresh_anonymized.csv"

if not os.path.exists(DATA_PATH):
    from google.colab import files
    uploaded = files.upload()          # a file picker appears: choose content_refresh_anonymized.csv
    DATA_PATH = list(uploaded.keys())[0]

df = pd.read_csv(DATA_PATH)

Saving content_refresh_anonymized.csv to content_refresh_anonymized.csv


In [4]:
import numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor

SEED = 42
DATA_PATH = "content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)

# Question: how many clicks will a page get in the latest 30 days,
# given what we knew BEFORE that window (prev 30d) plus static page facts?
TARGET    = "clicks_last_30d"
GROUP_COL = "client_id"

# Only "known before the window" features. The 90d totals, ctr, avg_position, trend_*,
# health_score and the is_*/needs_* flags all include or are built from the last 30 days
# (the target window), so they would leak the answer. Left out on purpose.
NUM = ["clicks_prev_30d", "impressions_prev_30d", "sessions_prev_30d",
       "search_volume", "competition", "cpc", "word_count", "char_count",
       "content_age_days", "days_since_last_update"]
CAT = ["content_type", "main_intent", "model_used"]

y_raw = df[TARGET]
y = np.log1p(y_raw)                       # clicks are very skewed: model log1p(clicks)
X = df[NUM + CAT].copy()
for c in ["clicks_prev_30d", "impressions_prev_30d", "sessions_prev_30d", "search_volume", "cpc"]:
    X[c] = np.log1p(X[c])                 # same log scale for the heavy-tailed inputs
for c in CAT:
    X[c] = X[c].fillna("missing").astype(str)

print("rows:", len(df), "| clients:", df[GROUP_COL].nunique())
print("share of pages with 0 clicks in target window:", round((y_raw == 0).mean(), 3))
print("missing share (top 5):\n", X.isna().mean().sort_values(ascending=False).head(5).round(3))
print("Spearman(prev clicks, target clicks):",
      round(df["clicks_prev_30d"].corr(y_raw, method="spearman"), 3))

def prep_linear():
    return ColumnTransformer([
        ("num", Pipeline([("i", SimpleImputer(strategy="median", add_indicator=True)),
                          ("s", StandardScaler())]), NUM),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)])

def prep_tree():
    return ColumnTransformer([
        ("num", SimpleImputer(strategy="median", add_indicator=True), NUM),
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CAT)])

rows: 30000 | clients: 32
share of pages with 0 clicks in target window: 0.612
missing share (top 5):
 word_count       0.257
char_count       0.257
cpc              0.082
search_volume    0.082
competition      0.082
dtype: float64
Spearman(prev clicks, target clicks): 0.769


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [5]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
tr_idx, te_idx = next(gss.split(X, y, groups=df[GROUP_COL]))
X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

shared = set(df.loc[tr_idx, GROUP_COL]) & set(df.loc[te_idx, GROUP_COL])
print("split: grouped by", GROUP_COL)
print("clients in train:", df.loc[tr_idx, GROUP_COL].nunique(), "| in test:", df.loc[te_idx, GROUP_COL].nunique())
print("clients shared between train and test:", len(shared), "(should be 0)")
print("train rows:", len(tr_idx), "| test rows:", len(te_idx))
print("mean log1p clicks train/test:", round(y_tr.mean(), 3), "/", round(y_te.mean(), 3))

split: grouped by client_id
clients in train: 24 | in test: 8
clients shared between train and test: 0 (should be 0)
train rows: 22885 | test rows: 7115
mean log1p clicks train/test: 0.74 / 0.506


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [6]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats import spearmanr
from sklearn.model_selection import GroupKFold

class Persistence:
    """Baseline: next window = same as previous window (clicks_prev_30d). No learning."""
    def fit(self, X, y): return self
    def predict(self, X): return X["clicks_prev_30d"].values       # already log1p

def make(est, pre): return Pipeline([("p", pre), ("m", est)])

models = {
    "Dummy (mean)":            make(DummyRegressor(strategy="mean"), prep_linear()),
    "Baseline (persistence)":  Persistence(),
    "Ridge":                   make(Ridge(alpha=1.0), prep_linear()),
    "Model (boosted trees)":   make(HistGradientBoostingRegressor(max_depth=4, learning_rate=0.05,
                                    max_iter=300, random_state=SEED), prep_tree()),
}
# NOTE: swap the "Baseline" for the exact rule/model from your Week-4 notebook if it differs.

rows, preds = [], {}
for name, m in models.items():
    m.fit(X_tr, y_tr)
    p = np.clip(m.predict(X_te), 0, None)
    preds[name] = p
    rows.append({"model": name,
                 "MAE (log1p)":   mean_absolute_error(y_te, p),
                 "MAE (clicks)":  mean_absolute_error(np.expm1(y_te), np.expm1(p)),
                 "R2 (log1p)":    r2_score(y_te, p),
                 "Spearman":      spearmanr(y_te, p)[0]})
results = pd.DataFrame(rows).set_index("model").round(3)
display(results)

b, m_ = results.loc["Baseline (persistence)"], results.loc["Model (boosted trees)"]
print(f"MAE (log1p) change vs baseline: {m_['MAE (log1p)'] - b['MAE (log1p)']:+.3f}  (negative = better)")
print("Observed on one client-grouped split only; directional, not a guarantee.")

# stability check: repeat the comparison across 5 client-grouped folds
fold = []
for f, (a, c) in enumerate(GroupKFold(n_splits=5).split(X, y, groups=df[GROUP_COL])):
    base = mean_absolute_error(y.iloc[c], X["clicks_prev_30d"].iloc[c])
    gb = make(HistGradientBoostingRegressor(max_depth=4, learning_rate=0.05, max_iter=300,
              random_state=SEED), prep_tree()).fit(X.iloc[a], y.iloc[a])
    fold.append({"fold": f, "baseline MAE": base,
                 "model MAE": mean_absolute_error(y.iloc[c], np.clip(gb.predict(X.iloc[c]), 0, None))})
fold = pd.DataFrame(fold).round(3)
fold["model better?"] = fold["model MAE"] < fold["baseline MAE"]
display(fold)

,MAE (log1p),MAE (clicks),R2 (log1p),Spearman
model,,,,
Dummy (mean),0.802,3.753,-0.060,NaN
Baseline (persistence),0.234,1.674,0.769,0.742
Ridge,0.277,1.891,0.785,0.675
Model (boosted trees),0.264,1.758,0.808,0.695


MAE (log1p) change vs baseline: +0.030  (negative = better)
Observed on one client-grouped split only; directional, not a guarantee.


,fold,baseline MAE,model MAE,model better?
0,0,0.329,0.328,True
1,1,0.406,0.423,False
2,2,0.297,0.308,False
3,3,0.198,0.272,False
4,4,0.207,0.226,False


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [7]:
from sklearn.inspection import permutation_importance

best = models["Model (boosted trees)"]
p = preds["Model (boosted trees)"]
err = df.iloc[te_idx][[GROUP_COL, "content_type", "clicks_prev_30d", "clicks_last_30d"]].copy()
err["pred_clicks"] = np.expm1(p).round(1)
err["abs_err_log"] = np.abs(y_te.values - p)

# 1) By volume band: where does it miss?
band = pd.cut(err["clicks_prev_30d"], [-1, 0, 2, 10, 50, np.inf], labels=["0", "1-2", "3-10", "11-50", "50+"])
by_band = err.groupby(band).agg(pages=("abs_err_log", "size"),
                                 avg_err_log=("abs_err_log", "mean"),
                                 actual_mean=("clicks_last_30d", "mean"),
                                 pred_mean=("pred_clicks", "mean")).round(2)
print("Error by prior-window click band:"); display(by_band)

# 2) By client: is the miss spread out or concentrated?
by_client = err.groupby(GROUP_COL).agg(pages=("abs_err_log", "size"),
                                        avg_err_log=("abs_err_log", "mean")).round(3)
print("Worst clients (test set):"); display(by_client.sort_values("avg_err_log", ascending=False).head(5))

# 3) By content type
print("Error by content type:"); display(err.groupby("content_type")["abs_err_log"].agg(["size", "mean"]).round(3))

# 4) What does it lean on? (permutation importance, drop in R2 when shuffled)
imp = permutation_importance(best, X_te, y_te, scoring="r2", n_repeats=5, random_state=SEED)
imp_df = (pd.DataFrame({"feature": X_te.columns, "importance": imp.importances_mean})
          .sort_values("importance", ascending=False).round(4))
print("Feature importance:"); display(imp_df)

# 5) Biggest misses, for reading by hand
print("Worst 5 misses:")
display(err.sort_values("abs_err_log", ascending=False).head(5).drop(columns=[GROUP_COL]))

Error by prior-window click band:


,pages,avg_err_log,actual_mean,pred_mean
clicks_prev_30d,,,,
0,4935,0.15,0.16,0.12
1-2,1042,0.48,1.41,0.74
3-10,699,0.57,5.57,3.46
11-50,359,0.54,22.08,15.23
50+,80,0.48,118.24,99.54


Worst clients (test set):


,pages,avg_err_log
client_id,,
client_8b940be7fb,28,0.572
client_4e07408562,2294,0.471
client_bdd2d3af3a,44,0.309
client_434c9b5ae5,87,0.303
client_f369cb89fc,1796,0.238


Error by content type:


,size,mean
content_type,,
comparison article,697,0.098
keyword article,6418,0.282


Feature importance:


,feature,importance
0,clicks_prev_30d,0.8679
1,impressions_prev_30d,0.1273
2,sessions_prev_30d,0.0073
12,model_used,0.0038
10,content_type,0.0008
3,search_volume,0.0005
11,main_intent,0.0003
4,competition,0.0003
7,char_count,0.0002
5,cpc,-0.0002


Worst 5 misses:


,content_type,clicks_prev_30d,clicks_last_30d,pred_clicks,abs_err_log
6325,keyword article,2,41,2.2,2.584891
28335,keyword article,169,13,147.9,2.364480
18870,keyword article,53,501,47.1,2.344872
4234,keyword article,1,17,0.8,2.288787
25364,keyword article,12,0,8.7,2.276633


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.